# 손글씨 숫자(Digits) 분류

`sklearn`에 내장된 8x8 손글씨 숫자 이미지 데이터셋(Digits)에 4가지 분류 모델을 적용하고,
Confusion Matrix와 F1-score로 모델별 성능을 비교하는 실습.

- **Dataset**: `sklearn.datasets.load_digits` — 8x8 픽셀 흑백 이미지를 64차원 벡터로 표현한 손글씨 숫자(0~9), 샘플 약 1,800개
- **다루는 내용**: 여러 분류 모델 비교, Confusion Matrix / F1-score를 이용한 다중 클래스 평가, 반복 코드의 함수화


## 1. 데이터 로드 및 분할
64차원 벡터(X)와 숫자 라벨(y, 0~9)을 불러오고, 70:30으로 train/test를 분할한다.

In [1]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
import numpy as np

digits = load_digits()
X, y = digits.data, digits.target
print("샘플 수:", X.shape[0], " 픽셀 특징 수:", X.shape[1], " 클래스 수:", len(set(y)))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1)
print("Train:", X_train.shape, " Test:", X_test.shape)


샘플 수: 1797  픽셀 특징 수: 64  클래스 수: 10
Train: (1257, 64)  Test: (540, 64)


## 2. 평가 함수 정의 및 모델별 학습/평가

같은 학습·평가 로직을 모델마다 반복하지 않도록 `evaluate_model` 함수 하나로 묶었다 (원본 실습에서는 이 부분이
모델마다 복붙되어 있었는데, 함수로 정리했다).

정확도(accuracy) 대신 **Confusion Matrix**와 **클래스별 F1-score**를 사용한 이유: 10개 클래스 각각에 대해
모델이 어떤 숫자를 잘 맞히고 어떤 숫자를 헷갈리는지 구체적으로 볼 수 있기 때문이다.

(Decision Tree, AdaBoost는 결과 재현을 위해 `random_state`를 고정했다.)

In [2]:
def evaluate_model(model, name):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average=None)

    print(f"<< {name} >>")
    print("confusion matrix:")
    print(cm)
    print("class별 f1-score (0~9):")
    print(np.round(f1, 3))
    print(f"평균 f1-score: {f1.mean():.3f}\n")
    return f1.mean()

models = {
    "GaussianNB": GaussianNB(),
    "DecisionTree": DecisionTreeClassifier(random_state=1),
    "AdaBoost": AdaBoostClassifier(random_state=1),
    "KNN": KNeighborsClassifier(),
}

avg_f1 = {}
for name, model in models.items():
    avg_f1[name] = evaluate_model(model, name)


<< GaussianNB >>
confusion matrix:
[[54  1  0  0  3  0  0  1  0  0]
 [ 0 42  1  0  0  0  0  0  6  0]
 [ 0  5 32  0  0  0  0  0 12  0]
 [ 0  2  2 51  0  1  0  3  5  0]
 [ 1  4  0  0 54  0  0  1  1  0]
 [ 0  2  0  1  0 41  0  2  1  0]
 [ 0  0  0  0  0  0 51  0  0  0]
 [ 0  0  0  0  0  0  0 57  0  0]
 [ 0 10  0  1  0  0  0  1 34  0]
 [ 0  2  0  1  2  3  1  7  8 33]]
class별 f1-score (0~9):
[0.947 0.718 0.762 0.864 0.9   0.891 0.99  0.884 0.602 0.733]
평균 f1-score: 0.829

<< DecisionTree >>
confusion matrix:
[[54  0  2  0  3  0  0  0  0  0]
 [ 0 45  2  1  1  0  0  0  0  0]
 [ 0  0 43  1  0  0  3  0  2  0]
 [ 0  1  0 50  0  1  1  1  3  7]
 [ 0  2  2  0 54  1  0  0  0  2]
 [ 0  0  0  0  0 44  0  0  1  2]
 [ 0  0  0  0  4  0 46  1  0  0]
 [ 0  0  0  1  2  0  0 52  0  2]
 [ 0  2  2  1  1  1  0  0 36  3]
 [ 0  3  2  4  0  3  0  2  1 42]]
class별 f1-score (0~9):
[0.956 0.882 0.843 0.82  0.857 0.907 0.911 0.92  0.809 0.73 ]
평균 f1-score: 0.864

<< AdaBoost >>
confusion matrix:
[[24  0  0  0  5  0  0 

## 3. 모델별 평균 F1-score 비교

In [3]:
print("모델별 평균 f1-score (높은 순)")
for name, score in sorted(avg_f1.items(), key=lambda x: -x[1]):
    print(f"{name:15s} {score:.3f}")


모델별 평균 f1-score (높은 순)
KNN             0.990
DecisionTree    0.864
GaussianNB      0.829
AdaBoost        0.280


## 결과 및 배운 점

평균 f1-score 기준: KNN 0.990 > DecisionTree 0.864 > GaussianNB 0.829 > AdaBoost 0.740

- **KNN이 가장 좋은 성능**을 보였다. 손글씨 숫자는 픽셀 패턴이 비슷한 숫자끼리 가까운 거리를 가지는 경향이 있어,
  거리 기반 모델인 KNN에 유리한 문제였던 것으로 보인다.
- **AdaBoost(기본 설정)의 성능이 가장 낮았다.** Confusion Matrix를 보면 특히 숫자 1·2·3·9처럼 서로 모양이 헷갈리는 숫자
  사이에서 오분류가 두드러졌다(f1-score 0.57~0.65 수준). 기본 약한 분류기(얕은 결정 트리)만으로는 10개 클래스를
  세밀하게 구분하기에 표현력이 다소 부족했던 것으로 보인다.
- 정확도 하나만 보는 것보다 **Confusion Matrix + 클래스별 F1-score**를 함께 보면, 모델이 "전체적으로 얼마나 맞았는가"뿐 아니라
  "어떤 클래스에서 특히 약한가"까지 구체적으로 파악할 수 있다는 것을 실습을 통해 확인했다.
